In [ ]:
from example_mortality_data import MALE_QX, FEMALE_QX

from example_cohorted_members import PARTICIPANTS_COHORTED
from example_unique_members import PARTICIPANTS_UNIQUE, PARTICIPANTS_UNIQUE_PI_SOLVED

In [ ]:
import numpy as np
import itertools
import matplotlib.pyplot as plt
from scipy.integrate import simpson, cumulative_trapezoid
from scipy.special import comb
import warnings

In [ ]:
"""
This script calculates participation rates (pi's) for a pool of people.

In case of single homogeneous cohort and gamma = 1 (log utility) the optimal d(t) = tpx / ax (we call this "natural payout function")

tpx = probability that a person who is alive at age x will still be alive t years later.
ax = Present Value of a life annuity that pays $1 per year, continuously, for as long as the person (aged x) stays alive.

In a very large (asymptotic) pool, a natural tontine behaves exactly like a standard fixed life annuity

In case of a heterogeneous tontine gamma = 1 the d(t) Milevsky & Salisbury (2016) propose is the weighted average of individual natural d(t)'s
-----------------------------------------------------------------------------------------------------------------------

For i cohorts in range 1...K:
K = number of cohorts
n_i = number of subscribers in cohort i
x_i = age of subscriber in i-th cohort
w_i = amount one wishes to invest

epv_i = one's Expected Present Value
lx = survival curve from qx
tpx interpolates lx by age

Gives everyone 1 pi, calculates epv_i, gives one with the "worst deal" boost to pi.
Repeats, until everyone's epv_i are equal (equitability).

Allows > 1 people in a cohort.
"""

def calculate_lx(qx_table):
    """
    Converts 'qx' (probability of dying) into 'lx' (number of people surviving).
    Starts with a base of 1.0 and reduces it year-by-year based on death probability.
    """
    lx = [1.0]
    for qx in qx_table:
        lx.append(lx[-1] * (1.0 - qx))
    return np.array(lx)

def run_combinatorial_state_solver():
    # --- Mortality Data Processing ---
    
    # Generate survival curves for both genders
    male_lx = calculate_lx(MALE_QX)
    female_lx = calculate_lx(FEMALE_QX)

    def tpx_empirical(age, t, gender):
        """
        Calculates the probability that a person of 'age' survives for 't' more years.
        Uses linear interpolation to handle non-integer ages or time steps.
        """
        lx_table = female_lx if gender == 'female' else male_lx
        # Find survival value at current age
        l_age = np.interp(age, np.arange(len(lx_table)), lx_table)
        # Find survival value at future age (age + t)
        l_age_t = np.interp(age + t, np.arange(len(lx_table)), lx_table)
        return l_age_t / l_age

    # --- System & Environment Setup ---

    interest_rate = 0.03            # Risk-free interest rate (3%) used for discounting future cash flows
    steps = 400         # Number of time increments for the simulation to integrate using Simpson's rule
    times = np.linspace(0, 100, steps)  # Projection horizon: 0 to 100 years

    # --- Participant Data Management ---

    n_participants = len(PARTICIPANTS_COHORTED)
    n_arr = np.array([p.get('n', 1) for p in PARTICIPANTS_COHORTED])
    invested_arr = np.array([p['invested'] for p in PARTICIPANTS_COHORTED])
    x_arr = np.array([p['age'] for p in PARTICIPANTS_COHORTED])
    g_arr = [p['gender'] for p in PARTICIPANTS_COHORTED]
    invested_total = np.sum(invested_arr * n_arr)

    # Pre-calculate survival probabilities for every cohort over the 'times' horizon
    tpx_matrix = np.array([tpx_empirical(x_arr[i], times, g_arr[i]) for i in range(n_participants)])

    # --- Mathematical Core Functions ---

    def calculate_ax(i, times):
        """
        Calculates the Life Annuity factor (a_x) for a cohort.
        This is the present value of $1 paid continuously while the person is alive.
        """
        discount = np.exp(-interest_rate * times)
        survival = tpx_matrix[i]
        return simpson(y=discount * survival, x=times)

    # Calculate annuity factors for all cohorts
    ax_arr = np.array([calculate_ax(i, times) for i in range(n_participants)])

    # Calculate the aggregate pool "dividend" density (d_t) over time
    d_t = np.zeros_like(times)
    for j in range(n_participants):
        alpha_j = (invested_arr[j] * n_arr[j]) / invested_total
        # d_t represents the mortality-weighted cash flow available to survivors
        d_t += alpha_j * (1.0 / ax_arr[j]) * tpx_matrix[j]

    def calculate_epv(pi, times, invested, x, d_t):
        """
        Calculates the Expected Present Value (epv) for each cohort given specific 
        participation rates (pi). This accounts for all possible survival combinations.
        """
        epv = np.zeros(n_participants)
        discount = np.exp(-interest_rate * times)
        
        # Pre-calculate Binomial distributions: 
        # What is the prob that 'k' people out of 'n' are alive at time 't'?
        prob_matrix = []
        for j in range(n_participants):
            n_j = n_arr[j]
            p = tpx_matrix[j]
            cohort_probs = []
            for k in range(n_j + 1):
                pk = comb(n_j, k) * (p**k) * ((1-p)**(n_j-k))
                cohort_probs.append(pk)
            prob_matrix.append(cohort_probs)

        # Generate all possible survival states (e.g., 0 survivors in group A, 2 in group B...)
        ranges = [range(n_j + 1) for n_j in n_arr]

        for i in range(n_participants):
            expected_share_t = np.zeros_like(times)
            
            for state in itertools.product(*ranges):
                # If the current person 'i' is dead in this state, they get $0
                if state[i] == 0:
                    continue
                
                # Calculate the joint probability of this specific survival state
                prob_state_t = np.ones_like(times)
                pool_value_t = 0.0

                for j in range(n_participants):
                    k = state[j]
                    if j == i:
                        # Adjust probability to represent a specific individual being alive
                        prob_state_t *= prob_matrix[j][k] * (k / n_arr[j])
                    else:
                        prob_state_t *= prob_matrix[j][k]
                    
                    # Total "claims" on the pool based on participation rates (pi)
                    pool_value_t += pi[j] * invested[j] * k

                # Add the individual's share of the pool for this state to the total expectation
                expected_share_t += prob_state_t * (pi[i] / pool_value_t)

            # Integrate over time to find the total Expected Present Value (epv) for the cohort
            integrand = discount * invested_total * d_t * expected_share_t
            epv[i] = simpson(y=integrand, x=times)
        return epv

    # --- 5. The Fairness Solver (Optimization) ---

    pi = np.ones(n_participants)  # Start with everyone having a rate of 1.0
    etas = [0.1, 0.01, 0.001, 0.0001, 0.00001] # Step sizes for refinement

    print("Starting solver...")

    for eta in etas:
        improved = True
        while improved:
            improved = False
            epv = calculate_epv(pi, times, invested_arr, x_arr, d_t)
            # The 'target' is the weighted average return of the whole pool
            target = np.sum((invested_arr * n_arr / invested_total) * epv) 

            for i in range(n_participants):
                # If a cohort's return is below average, increase their participation rate (pi)
                if epv[i] < target:
                    pi_test = pi.copy()
                    pi_test[i] += eta
                    epv_test = calculate_epv(pi_test, times, invested_arr, x_arr, d_t)
                    target_test = np.sum((invested_arr * n_arr / invested_total) * epv_test)
                    
                    # Check if the adjustment actually moved them closer to equity
                    if epv_test[i] < target_test:
                        pi = pi_test
                        improved = True

    # --- Results Output ---

    # Normalize so the lowest participation rate is 1.0
    pi_normalized = pi / np.min(pi)
    final_epv = calculate_epv(pi_normalized, times, invested_arr, x_arr, d_t)

    print("\n--- Equitable Share Prices Found ---")
    for i in range(n_participants):
        print(f"\nCohort {i} (Count: {n_arr[i]}, Age {x_arr[i]}, Invests ${invested_arr[i]} each):")
        print(f"  Share Price (1/pi):                {1.0 / pi_normalized[i]:.5f}")
        print(f"  Participation Rate (pi):           {pi_normalized[i]:.5f}")
        print(f"  Present Value return (epv_i):      {final_epv[i]:.5f}")

    inequity = np.max(final_epv) - np.min(final_epv)
    print(f"\nFinal Inequity Spread: {inequity:.10f} (Closer to 0 is better)")

    print("\nParticipation Rates (pi) as a Python list:")
    print("[")
    for i, p in enumerate(pi_normalized):
        # Adds a comma to every line except the last one
        comma = "," if i < len(pi_normalized) - 1 else ""
        print(f"    {p:.20f}{comma}")
    print("]")

if __name__ == "__main__":
    run_combinatorial_state_solver()

In [ ]:
"""
This script calculates participation rates (pi's) for a pool of people.

In case of single homogeneous cohort and gamma = 1 (log utility) then the optimal d(t) = tpx / ax (we call this "natural payout function")

tpx = probability that a person who is alive at age x will still be alive t years later.
ax = Present Value of a life annuity that pays $1 per year, continuously, for as long as the person (aged x) stays alive.

In a very large (asymptotic) pool, a natural tontine behaves exactly like a standard fixed life annuity

In case of a heterogeneous tontine gamma = 1 the d(t) Milevsky & Salisbury (2016) propose is the weighted average of individual natural d(t)'s
-----------------------------------------------------------------------------------------------
The Delta Method (specifically the 2nd-order Taylor expansion used in code) follows this logic:

STEP 1. Start with the "Share of the Average":
Calculate the share as if everyone survived exactly according to their life expectancy

claim_weight / (claim_weight + mu)

claim_weight_i ("Person i's Claim if they survive") = invested_i * pi_i

mu ("Average amount of other claims expected to be alive.") = sum(
Claim_j * p_j
)

Person i is excluded from j - not to count him twice!!

STEP 2. Add a Variance Correction: 
Because the "Average Share" is usually higher than the "Share of the Average"
(Jensen's inequality),
We add:

(claim_weight * sigma ** 2) / (claim_weight + mu) ** 3

sigma ("Variance from mu") = sum(
Claim_j ** 2 * p_j * (1 - p_j)
)

Person i is excluded from j - not to count him twice!!

At the end of that get a single number, like 0.025.
Meaning: "At Year 12, if you are alive, you are expected to receive 2.5% of the total money being distributed."

We then:
Take that 0.025 and multiply it by the probability that an annuitant is actually alive to collect it.
Multiply by the total Pot - d(t) * invested_total.
Discount by interest rate.
Integrate the sum (see "steps" below).

That is total dollar epv_i; we report epv_i / invested_i (per $1 invested). We adjust pi_i's until everyone's epv_i are virtually equal.
"""

def run_delta_method_pi_approximator():
    def calculate_lx(qx_table):
        """Converts a QX table to a radix-based survival table l_x."""
        lx = [1.0]
        for qx in qx_table:
            lx.append(lx[-1] * (1.0 - qx))
        return np.array(lx)

    male_lx = calculate_lx(MALE_QX)
    female_lx = calculate_lx(FEMALE_QX)

    def tpx_empirical(age, t, gender):
        """Calculates survival probability using linear interpolation on l_x."""
        lx_table = female_lx if gender == 'female' else male_lx
        # Survival from birth to age
        l_age = np.interp(age, np.arange(len(lx_table)), lx_table)
        # Survival from birth to age + t
        l_age_t = np.interp(age + t, np.arange(len(lx_table)), lx_table)
        return l_age_t / l_age

    # --- 3. System Parameters ---
    interest_rate = 0.03       # Risk-free interest rate
    steps = 400   # Simpson's rule time steps (how finely we slice the integration)
    times = np.linspace(0, 150, steps)

    # --- 4. Participant Data ---

    n_participants = len(PARTICIPANTS_UNIQUE)
    invested_arr = np.array([p['invested'] for p in PARTICIPANTS_UNIQUE])
    x_arr = np.array([p['age'] for p in PARTICIPANTS_UNIQUE])
    g_arr = [p['gender'] for p in PARTICIPANTS_UNIQUE]
    invested_total = np.sum(invested_arr)

    # Pre-calculate survival vectors for each participant across the 'times' grid
    tpx_matrix = np.array([tpx_empirical(x_arr[i], times, g_arr[i]) for i in range(n_participants)])

    # --- 5. Core Functions ---
    def calc_ax(i, times):
        """Calculates standard annuity price a_x using pre-calculated survival."""
        discount = np.exp(-interest_rate * times)
        survival = tpx_matrix[i]
        return simpson(y=discount * survival, x=times)

    # Calculate base annuity factors and the Proportional Tontine d(t) 
    ax_arr = np.array([calc_ax(i, times) for i in range(n_participants)])
    d_t = np.zeros_like(times)
    for j in range(n_participants):
        alpha_j = invested_arr[j] / invested_total
        d_t += alpha_j * (1.0 / ax_arr[j]) * tpx_matrix[j]

    def calculate_epv(pi, times, invested, x, d_t):
        """EPV per $1 invested (dimensionless), consistent with the combinatorial solver."""
        epv = np.zeros(n_participants)
        discount = np.exp(-interest_rate * times)
        
        weights = pi * invested
        mu_total = np.zeros_like(times)
        var_total = np.zeros_like(times)

        # 1. Precalculate the Mean and Variance for the entire pool
        for j in range(n_participants):
            p_alive = tpx_matrix[j]
            w_j = weights[j]
            mu_total += w_j * p_alive
            var_total += (w_j ** 2) * p_alive * (1.0 - p_alive)

        for i in range(n_participants):
            claim_weight = weights[i]
            p_alive_i = tpx_matrix[i]
            
            mu_minus_i = mu_total - (claim_weight * p_alive_i)
            var_minus_i = var_total - ((claim_weight ** 2) * p_alive_i * (1.0 - p_alive_i))
            
            var_minus_i = np.maximum(var_minus_i, 0.0)

            expected_share_t = (claim_weight / (claim_weight + mu_minus_i)) + (claim_weight * var_minus_i) / ((claim_weight + mu_minus_i) ** 3)

            integrand = discount * tpx_matrix[i] * invested_total * d_t * expected_share_t
            epv[i] = simpson(y=integrand, x=times) / invested[i]
            
        return epv

    # --- 6. Iterative Solver ---
    pi = np.ones(n_participants) 
    etas = [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001]

    print("Starting solver...")
    for eta in etas:
        improved = True
        while improved:
            improved = False
            epv = calculate_epv(pi, times, invested_arr, x_arr, d_t)
            target = np.sum((invested_arr / invested_total) * epv) 

            for i in range(n_participants):
                if epv[i] < target:
                    pi_test = pi.copy()
                    pi_test[i] += eta
                    epv_test = calculate_epv(pi_test, times, invested_arr, x_arr, d_t)
                    target_test = np.sum((invested_arr / invested_total) * epv_test)
                    if epv_test[i] < target_test:
                        pi = pi_test
                        improved = True

    pi_normalized = pi / np.min(pi)
    final_epv = calculate_epv(pi_normalized, times, invested_arr, x_arr, d_t)

    print("\n--- Equitable Share Prices Found ---")
    for i in range(n_participants):
        print(f"\nParticipant {i} (Age {x_arr[i]}, Invests ${invested_arr[i]}):")
        print(f"  Share Price (1/pi):           {1.0 / pi_normalized[i]:.4f}")
        print(f"  Participation Rate (pi):      {pi_normalized[i]:.4f}")
        print(f"  Present Value return (epv_i): {final_epv[i]:.5f}")

    inequity = np.max(final_epv) - np.min(final_epv)
    print(f"\nFinal Inequity Spread: {inequity:.10f} (Closer to 0 is better)")

    print("\nParticipation Rates (pi) as a Python list:")
    print("[")
    for i, p in enumerate(pi_normalized):
        # Adds a comma to every line except the last one
        comma = "," if i < len(pi_normalized) - 1 else ""
        print(f"    {p:.20f}{comma}")
    print("]")

if __name__ == "__main__":
    run_delta_method_pi_approximator()

In [ ]:
"""
This script models the stochastic performance of a fair tontine pool.

It is FULL MONTE CARLO and can be used with large amounts of participants.

It visualizes participant cumulative returns and their variance, shares outstanding with their variance, 
shared payout schedule and per-year payouts with their variance.

Important: in case of a positive discount rate, the younger members are expected to gain more
in terms of absolute amount, that's why the second plot accounts for time value of money.

"Immortality" is given to members one by one to optimize data collection for the conditional plot.

It uses cumulative_trapezoid as opposed to Simpson's rule in the solver.
"""

# --- Mortality & Actuarial Foundations ---
def calculate_lx(qx_table):
    """Converts a sequence of mortality probabilities (qx) into a survival curve (lx)."""
    lx = [1.0]
    for qx in qx_table:
        lx.append(lx[-1] * (1.0 - qx))
    return np.array(lx)

# Survival curves derived from mortality tables
male_lx = calculate_lx(MALE_QX)
female_lx = calculate_lx(FEMALE_QX)

def tpx_empirical(age, t, gender):
    """Calculates the probability of an individual aged 'x' surviving 't' years (tPx)."""
    lx_table = female_lx if gender == 'female' else male_lx
    l_age = np.interp(age, np.arange(len(lx_table)), lx_table)
    l_age_t = np.interp(age + t, np.arange(len(lx_table)), lx_table)
    return np.where(l_age == 0, 0.0, l_age_t / l_age)

# --- Simulation and Discretization Settings ---
interest_rate = 0.03
steps = 400 # Temporal resolution of the model (total number of data points over the timeframe).
times = np.linspace(0, 150, steps) # Generates a linear time grid from year 0 to year 100, used for survival indexing and integration.
dt = times[1] - times[0] # Calculates the infinitesimal time increment (delta t) required for accurate numerical integration.
N_CUM_SIMS = 10_000

# --- Pool Initialization --

n_participants = len(PARTICIPANTS_UNIQUE_PI_SOLVED)
invested_arr = np.array([p['invested'] for p in PARTICIPANTS_UNIQUE_PI_SOLVED])
x_arr = np.array([p['age'] for p in PARTICIPANTS_UNIQUE_PI_SOLVED])
g_arr = [p['gender'] for p in PARTICIPANTS_UNIQUE_PI_SOLVED]
invested_total = np.sum(invested_arr)

# Pre-calculate tPx for all participants across the time horizon
tpx_matrix = np.array([tpx_empirical(x_arr[i], times, g_arr[i]) for i in range(n_participants)])

# --- Analytical (Expectation-Based) Framework ---
def calc_ax(i, times):
    """Calculates the actuarial present value of a life annuity of 1 per year (ax)."""
    return simpson(y=np.exp(-interest_rate * times) * tpx_matrix[i], x=times)

pi_normalized = np.array([p['pi'] for p in PARTICIPANTS_UNIQUE_PI_SOLVED])
shares_arr = pi_normalized * invested_arr
ax_arr = np.array([calc_ax(i, times) for i in range(n_participants)])

# Determine the aggregate payout rate d(t) based on the reciprocal of life expectancy at entry
d_t = np.zeros_like(times)
for j in range(n_participants):
    alpha_j = invested_arr[j] / invested_total
    d_t += alpha_j * (1.0 / ax_arr[j]) * tpx_matrix[j]

# --- Stochastic Simulation Engine ---
def run_simulation(pi, d_t, times, immortal_idx=None):
    """Simulates a single pool realization by drawing random death times for participants."""
    n = len(x_arr)
    shares = pi * invested_arr
    death_times = []
    for i in range(n):
        if i == immortal_idx:
            death_times.append(times[-1] + 1) # Force survival for conditional analysis
        else:
            random_roll = np.random.random()
            death_idx = np.where(tpx_matrix[i] < random_roll)[0]
            death_times.append(times[death_idx[0]] if len(death_idx) > 0 else times[-1])
            
    cum_divs = np.zeros((n, len(times)))
    disc_cum_divs = np.zeros((n, len(times)))
    share_history = np.zeros(len(times))
    payout_rates = np.full((n, len(times)), np.nan)
    
    # Iterate through time to determine payouts based on surviving shares
    for t_idx, t in enumerate(times):
        is_alive = np.array([death_times[i] > t for i in range(n)])
        current_shares = shares * is_alive
        sum_shares = np.sum(current_shares)
        share_history[t_idx] = sum_shares
        
        if sum_shares > 0:
            total_payout_rate = invested_total * d_t[t_idx]
            for i in range(n):
                if is_alive[i]:
                    payout_rates[i, t_idx] = shares[i] * (total_payout_rate / sum_shares)
                    prev_val = cum_divs[i, t_idx-1] if t_idx > 0 else 0
                    cum_divs[i, t_idx] = prev_val + (payout_rates[i, t_idx] * dt)
                    disc_prev = disc_cum_divs[i, t_idx-1] if t_idx > 0 else 0
                    disc_cum_divs[i, t_idx] = disc_prev + np.exp(-interest_rate * t) * payout_rates[i, t_idx] * dt
                else:
                    cum_divs[i, t_idx] = cum_divs[i, t_idx-1] if t_idx > 0 else 0
                    disc_cum_divs[i, t_idx] = disc_cum_divs[i, t_idx-1] if t_idx > 0 else 0
        elif t_idx > 0:
            cum_divs[:, t_idx] = cum_divs[:, t_idx-1]
            disc_cum_divs[:, t_idx] = disc_cum_divs[:, t_idx-1]
    return cum_divs, share_history, payout_rates, disc_cum_divs

# --- Standard Monte Carlo Execution ---
# Memory-safe running sums instead of storing every simulation
sum_cum_divs = np.zeros((n_participants, len(times)))
sq_sum_cum_divs = np.zeros((n_participants, len(times)))
sum_disc_cum_divs = np.zeros((n_participants, len(times)))
sq_sum_disc_cum_divs = np.zeros((n_participants, len(times)))

sum_share_history = np.zeros(len(times))
sq_sum_share_history = np.zeros(len(times))

sum_payout_rates = np.zeros((n_participants, len(times)))
valid_payout_count = np.zeros((n_participants, len(times)))

print(f"Running {N_CUM_SIMS} standard simulations...")
for sim in range(N_CUM_SIMS):
    c_d, s_h, p_r, dc_d = run_simulation(pi_normalized, d_t, times)
    
    sum_cum_divs += c_d
    sq_sum_cum_divs += c_d ** 2
    sum_disc_cum_divs += dc_d
    sq_sum_disc_cum_divs += dc_d ** 2
    
    sum_share_history += s_h
    sq_sum_share_history += s_h ** 2
    
    # Safely accumulate only the non-NaN payout rates
    valid_mask = ~np.isnan(p_r)
    sum_payout_rates[valid_mask] += p_r[valid_mask]
    valid_payout_count += valid_mask.astype(int)

# Process stochastic results into means and standard deviations algebraically
mean_sim_cum_divs = sum_cum_divs / N_CUM_SIMS
var_sim_cum_divs = (sq_sum_cum_divs / N_CUM_SIMS) - (mean_sim_cum_divs ** 2)
std_sim_cum_divs = np.sqrt(np.maximum(var_sim_cum_divs, 0))

mean_sim_disc_cum_divs = sum_disc_cum_divs / N_CUM_SIMS
var_sim_disc_cum_divs = (sq_sum_disc_cum_divs / N_CUM_SIMS) - (mean_sim_disc_cum_divs ** 2)
std_sim_disc_cum_divs = np.sqrt(np.maximum(var_sim_disc_cum_divs, 0))

mean_sim_share_history = sum_share_history / N_CUM_SIMS
var_sim_share_history = (sq_sum_share_history / N_CUM_SIMS) - (mean_sim_share_history ** 2)
std_sim_share_history = np.sqrt(np.maximum(var_sim_share_history, 0))

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    mean_sim_payout_rate = np.divide(sum_payout_rates, valid_payout_count, 
                                     out=np.full_like(sum_payout_rates, np.nan), 
                                     where=valid_payout_count!=0)

# --- Conditional Monte Carlo Execution ---
# Calculates volatility for an individual specifically while they are still alive

N_COND_SIMS = 500

cond_mean_payout_rate = np.zeros((n_participants, len(times)))
cond_std_payout_rate = np.zeros((n_participants, len(times)))

print(f"Running {N_COND_SIMS} conditional simulations per participant for subplot 3...")
for i in range(n_participants):
    # Memory-safe running sums for the specific immortal participant
    sum_part_payouts = np.zeros(len(times))
    sq_sum_part_payouts = np.zeros(len(times))
    
    for sim in range(N_COND_SIMS):
        _, _, p_r, _ = run_simulation(pi_normalized, d_t, times, immortal_idx=i)
        p_r_i = p_r[i]
        
        sum_part_payouts += p_r_i
        sq_sum_part_payouts += p_r_i ** 2
        
    mean_p = sum_part_payouts / N_COND_SIMS
    var_p = (sq_sum_part_payouts / N_COND_SIMS) - (mean_p ** 2)
    cond_mean_payout_rate[i] = mean_p
    cond_std_payout_rate[i] = np.sqrt(np.maximum(var_p, 0))

# --- Visualization ---
plt.style.use('dark_background')
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 24))

cmap_sim = plt.get_cmap('BuPu')
cmap_anal = plt.get_cmap('Purples')

colors_sim = [cmap_sim(0.3 + (i / (n_participants * 1.5))) for i in range(n_participants)]
colors_anal = [cmap_anal(0.4 + (i / (n_participants * 1.5))) for i in range(n_participants)]

is_data_present = ~np.isnan(mean_sim_payout_rate)
max_time_to_show = times[np.where(is_data_present)[1].max()] if np.any(is_data_present) else times[-1]
plot_limit = min(np.ceil(max_time_to_show / 5) * 5, times[-1])
tick_spacing = np.arange(0, plot_limit + 5, 5)

seen_ages_ax1 = set()
for i in range(n_participants):
    age = x_arr[i]
    label_fill = f"1$\sigma$ ({age} y.o.)" if age not in seen_ages_ax1 else "_nolegend_"
    label_line = f"Sim. Mean ({age} y.o.)" if age not in seen_ages_ax1 else "_nolegend_"
    
    ax1.fill_between(times, 
                     mean_sim_cum_divs[i] - std_sim_cum_divs[i], 
                     mean_sim_cum_divs[i] + std_sim_cum_divs[i], 
                     color=colors_sim[i], alpha=0.1, label=label_fill)
    ax1.plot(times, mean_sim_cum_divs[i], color=colors_anal[i], lw=2, label=label_line)
    seen_ages_ax1.add(age)

ax1.set_xlim(0, plot_limit)
ax1.set_xticks(tick_spacing)
ax1.set_ylabel("Cumulative Dividends ($)")
ax1.set_xlabel("Years Elapsed")
ax1.set_title("Mean of Cumulative Payouts with 1 Std Dev")
ax1.grid(True, alpha=0.2)
ax1.legend(loc='upper left', fontsize='x-small', ncol=2)

seen_ages_ax2 = set()
for i in range(n_participants):
    age = x_arr[i]
    label_fill = f"1$\sigma$ ({age} y.o.)" if age not in seen_ages_ax2 else "_nolegend_"
    label_line = f"Sim. Mean ({age} y.o.)" if age not in seen_ages_ax2 else "_nolegend_"
    ax2.fill_between(times,
                     mean_sim_disc_cum_divs[i] - std_sim_disc_cum_divs[i],
                     mean_sim_disc_cum_divs[i] + std_sim_disc_cum_divs[i],
                     color=colors_sim[i], alpha=0.1, label=label_fill)
    ax2.plot(times, mean_sim_disc_cum_divs[i], color=colors_anal[i], lw=2, label=label_line)
    seen_ages_ax2.add(age)

ax2.set_xlim(0, plot_limit)
ax2.set_xticks(tick_spacing)
ax2.set_ylabel("Discounted Cumulative Dividends ($)")
ax2.set_xlabel("Years Elapsed")
ax2.set_title(f"Mean of Discounted Cumulative Payouts (r={interest_rate*100:.2f}%) with 1 Std Dev")
ax2.grid(True, alpha=0.2)
ax2.legend(loc='upper left', fontsize='x-small', ncol=2)

def get_padded_limits(data, padding=0.1):
    d_min, d_max = np.nanmin(data), np.nanmax(data)
    d_range = d_max - d_min
    return d_min - padding * d_range, d_max + padding * d_range

dash_pattern = (5, 5)

ax3.plot(times, mean_sim_share_history, color='steelblue', lw=2,
         linestyle=(0, dash_pattern), label="Mean Shares Outstanding")

ax3.fill_between(times,
                 mean_sim_share_history - std_sim_share_history,
                 mean_sim_share_history + std_sim_share_history,
                 color='steelblue', alpha=0.2, label="1 Std Dev of Mean Shares Outstanding")
ax3.set_ylabel("Mean Shares Outstanding", color='steelblue')
ax3.set_xlabel("Years Elapsed")

ax3_twin = ax3.twinx()
ax3_twin.plot(times, d_t, color='firebrick', lw=2,
              linestyle=(5, dash_pattern), label="d(t)")
ax3_twin.set_ylabel("Payout Rate d(t)", color="firebrick")

padding_factor = 0.15
ax3.set_ylim(*get_padded_limits(mean_sim_share_history, padding_factor))
ax3_twin.set_ylim(*get_padded_limits(d_t, padding_factor))

ax3.set_xlim(0, plot_limit)
ax3.set_xticks(tick_spacing)
ax3.set_title("Mean Shares Outstanding (Monte Carlo) with 1 Std Dev, Payout Rate d(t)")
ax3.grid(True, alpha=0.3)

h1, l1 = ax3.get_legend_handles_labels()
h2, l2 = ax3_twin.get_legend_handles_labels()
ax3.legend(h1+h2, l1+l2, loc='upper right', fontsize='small')

seen_ages_ax4 = set()
for i in range(n_participants):
    age = x_arr[i]
    label_fill = f"1$\sigma$ ({age} y.o.)" if age not in seen_ages_ax4 else "_nolegend_"
    label_line = f"Sim. Mean ({age} y.o.)" if age not in seen_ages_ax4 else "_nolegend_"
    ax4.fill_between(times,
                     cond_mean_payout_rate[i] - cond_std_payout_rate[i],
                     cond_mean_payout_rate[i] + cond_std_payout_rate[i],
                     color=colors_sim[i], alpha=0.1, label=label_fill)
    ax4.plot(times, cond_mean_payout_rate[i], color=colors_anal[i], lw=2, label=label_line)
    seen_ages_ax4.add(age)

ax4.set_xlim(0, plot_limit)
ax4.set_xticks(tick_spacing)
ax4.set_ylabel("Payout Rate ($/year)")
ax4.set_xlabel("Years Elapsed")
ax4.set_title("Mean Conditional Annual Payout Rate: as Long as Member is Alive with 1 Std Dev")
ax4.grid(True, alpha=0.2)
ax4.legend(loc='upper right', fontsize='x-small', ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
"""
Lifetime expected utility for each tontine member (log utility, gamma=1).

Inputs (from PARTICIPANTS_UNIQUE_PI_SOLVED):
age, gender, invested, pi - pi already set by the fairness solver elsewhere.

Main symbols:
tpx     P(alive t years from now)
ax      PV of $1/yr while alive  = ∫ e^{-rt} tpx dt
d(t)    Pool payout rate         = Σ (invested_j / total) x (tpx_j / ax_j)
w_j     Claim weight             = pi_j x invested_j
interest_rate =     Discount (interest) rate

Steps:
1. lx = survival curve from qx, tpx interpolates lx by age
2. Build d(t): capital-weighted blend of each member's natural payout (tpx/ax).
3. At each t, pool "competing members alive" μ = Σ w_j·tpx_j; variance from independent deaths.
4. For member i (if alive): share ≈ w_i / (w_i + μ_{-i}); payout ≈ share x total x d(t).
5. E[log(payout)] ≈ log(mean payout) + 0.5·Var(share denominator) — delta shortcut.
6. Utility_i = ∫ e^{-rt} · tpx_i · E[log payout at t] dt  (Simpson over times).

Output: one utility number per person. Compare pools/members; higher = better under log utility.
Not solving pi here — only scoring the pool given solved pi.
"""

def calculate_lx(qx_table):
    lx = [1.0]
    for qx in qx_table:
        lx.append(lx[-1] * (1.0 - qx))
    return np.array(lx)

male_lx = calculate_lx(MALE_QX)
female_lx = calculate_lx(FEMALE_QX)

def tpx_empirical(age, t, gender):
    lx_table = female_lx if gender == 'female' else male_lx
    l_age = np.interp(age, np.arange(len(lx_table)), lx_table)
    l_age_t = np.interp(age + t, np.arange(len(lx_table)), lx_table)
    return l_age_t / l_age

# --- SETUP AND INPUT DATA ---
# Set the interest rate and create a timeline from year 0 to 100
interest_rate = 0.03
steps = 400
times = np.linspace(0, 100, steps)

# Convert the list of dictionaries into NumPy arrays for faster mathematical operations
n_participants = len(PARTICIPANTS_UNIQUE_PI_SOLVED)
invested_arr = np.array([p['invested'] for p in PARTICIPANTS_UNIQUE_PI_SOLVED])
pi_arr = np.array([p['pi'] for p in PARTICIPANTS_UNIQUE_PI_SOLVED])
x_arr = np.array([p['age'] for p in PARTICIPANTS_UNIQUE_PI_SOLVED])
g_arr = [p['gender'] for p in PARTICIPANTS_UNIQUE_PI_SOLVED]
invested_total = np.sum(invested_arr)

# --- ACTUARIAL CALCULATIONS ---
# Create a matrix where each row represents a person's probability of being alive at each point in 'times'
tpx_matrix = np.array([tpx_empirical(x_arr[i], times, g_arr[i]) for i in range(n_participants)])

# Function to calculate the 'annuity factor' (ax) for each person using Simpson's Rule for integration
def calculate_ax(i, times):
    discount = np.exp(-interest_rate * times)
    return simpson(y=discount * tpx_matrix[i], x=times)

ax_arr = np.array([calculate_ax(i, times) for i in range(n_participants)])

# Determine the total payout of the tontine over time based on the survival of the group
d_t = np.zeros_like(times)
for j in range(n_participants):
    alpha_j = invested_arr[j] / invested_total
    d_t += alpha_j * (1.0 / ax_arr[j]) * tpx_matrix[j]

# --- UTILITY AND RISK ANALYSIS ---
def calculate_individual_utilities():
    utilities = np.zeros(n_participants)
    discount = np.exp(-interest_rate * times)
    
    # Calculate weighted participation and initialize pool-wide survival statistics
    weights = pi_arr * invested_arr
    mu_total = np.zeros_like(times)
    var_total = np.zeros_like(times)

    # Calculate the expected number of survivors (mu) and the variance in the pool
    for j in range(n_participants):
        p_alive = tpx_matrix[j]
        w_j = weights[j]
        mu_total += w_j * p_alive
        var_total += (w_j ** 2) * p_alive * (1.0 - p_alive)

    # Loop through each person to calculate their specific expected utility
    for i in range(n_participants):
        claim_weight = weights[i]
        p_alive_i = tpx_matrix[i]
        
        mu_minus_i = mu_total - (claim_weight * p_alive_i)
        var_minus_i = np.maximum(var_total - ((claim_weight ** 2) * p_alive_i * (1.0 - p_alive_i)), 0.0)
        
        mean_share = claim_weight / (claim_weight + mu_minus_i)
        consumption = mean_share * invested_total * d_t
        
        safe_consumption = np.maximum(consumption, 1e-10)
        log_u_mean = np.log(safe_consumption)
        
        denom = np.maximum((claim_weight + mu_minus_i)**2, 1e-10)
        variance_correction = 0.5 * var_minus_i / denom
        
        # Combine the log utility and the variance correction for the expected utility at time 't'
        expected_utility_t = log_u_mean + variance_correction
        
        # Weight the utility by the probability the person is actually alive and the time-value of money
        integrand = discount * p_alive_i * expected_utility_t
        
        # Replace any mathematical errors (NaN/Infinity) with zero to keep the code running
        integrand = np.nan_to_num(integrand, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Use Simpson's Rule to sum (integrate) these values over the entire 100-year timeline
        utilities[i] = simpson(y=integrand, x=times)
        
    return utilities

# --- EXECUTION ---
results = calculate_individual_utilities()

print("--- Calculated Utilities Per Member ---")
for i in range(n_participants):
    print(f"Participant {i} (Age: {x_arr[i]}, Invested: {invested_arr[i]}, Gender: {g_arr[i]}): Utility = {results[i]:.6f}")